# OmniVoice REST API — Kaggle TTS Backend

Exposes long-form voice cloning as a stable REST endpoint (fixed URL via a reserved
ngrok domain) for the explainer-video pipeline to call directly. No manual UI step —
run all cells, copy the printed URL into `pipeline/.env` (`OMNIVOICE_BASE`) once.

**Before running:** turn the **Internet** toggle ON in the right-hand *Session Options*
sidebar, and set the `NGROK_AUTHTOKEN` / `NGROK_DOMAIN` secrets under *Add-ons → Secrets*.

In [ ]:
import os
from pathlib import Path

# 1. Map Hugging Face cache to Kaggle's spacious writable directory
os.environ["HF_HOME"] = "/kaggle/working/hf_home"
os.environ["TRANSFORMERS_CACHE"] = "/kaggle/working/hf_home/transformers"
os.environ["HF_HUB_CACHE"] = "/kaggle/working/hf_home/hub"

Path(os.environ["HF_HOME"]).mkdir(parents=True, exist_ok=True)
print("✅ HF cache successfully mapped to:", os.environ["HF_HOME"])

# 1b. Load HF_TOKEN from Kaggle Secrets to bypass downloading rate-limits
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        print("🔑 HF_TOKEN successfully loaded from Kaggle Secrets!")
    else:
        print("⚠️ HF_TOKEN not found in Kaggle Secrets. Running without authentication (might get throttled).")
except Exception as e:
    print("⚠️ Could not access Kaggle Secrets. Running unauthenticated.")

# 2. Install required package dependencies
print("📥 Installing omnivoice, soundfile, fastapi, uvicorn, pyngrok packages...")
!pip install -q omnivoice soundfile fastapi "uvicorn[standard]" pyngrok nest_asyncio

# 3. Load the model weights and set compute device
print("🚀 Initializing model (Downloading ~3-5 GB weights on first launch)...")
import torch
import torchaudio
import re
import numpy as np
import soundfile as sf
from omnivoice import OmniVoice

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
dtype  = torch.float16 if torch.cuda.is_available() else torch.float32
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)')

model = OmniVoice.from_pretrained('k2-fsa/OmniVoice', device_map=device, dtype=dtype)
print('✅ OmniVoice model loaded successfully!')

---
## 🛠️ Automated Production Pipeline for Long-Form Voiceovers (45+ Mins)
The function below resolves the hallucination and pacing issues when handling massive text streams. It automatically breaks your text down into manageable sentence blocks, generates each sentence using stable compute windows, injects structural phrasing pauses (`pause_duration`), and knits them into a single high-quality master file.

In [ ]:
def generate_long_voiceover(model_instance, script_text, ref_audio_path=None, output_filename="/kaggle/working/master_voiceover.wav", speed=0.88, num_steps=32, pause_duration=0.4):
    """
    Automates rendering massive script layouts safely into unified, studio-grade audio.
    """
    sentence_chunks = re.split(r'(?<=[.!?])\s+', script_text.strip())
    sentence_chunks = [c.strip() for c in sentence_chunks if c.strip()]
    
    if not sentence_chunks:
        print("❌ Error: Script content appears to be empty.")
        return
        
    print(f"🤖 Script successfully initialized. Processing {len(sentence_chunks)} structural chunks safely...")
    
    sample_rate = 24000
    all_audio_segments = []
    
    pause_buffer = np.zeros(int(sample_rate * pause_duration), dtype=np.float32)
    
    for idx, chunk in enumerate(sentence_chunks):
        print(f"🎙️ Generating block {idx+1}/{len(sentence_chunks)}: \"{chunk[:45]}...\"")
        
        try:
            kwargs = {
                "text": chunk,
                "num_step": int(num_steps),
                "speed": float(speed)
            }
            
            if ref_audio_path and os.path.exists(ref_audio_path):
                kwargs["ref_audio"] = ref_audio_path
            
            output = model_instance.generate(**kwargs)
            audio_data = output[0]
            
            if torch.is_tensor(audio_data):
                audio_data = audio_data.detach().cpu().numpy()
                
            audio_data = audio_data.flatten()
            
            all_audio_segments.append(audio_data)
            all_audio_segments.append(pause_buffer)
            
        except Exception as e:
            print(f"⚠️ Error processing block {idx+1}: {e}. Skipping to maintain execution loop flow.")
            continue
            
    if all_audio_segments:
        print("\n🪡 Stitching segments seamlessly into production file output...")
        final_audio = np.concatenate(all_audio_segments)
        sf.write(output_filename, final_audio, sample_rate)
        print(f"🎉 Generation Completed! File stored at: {output_filename}")
    else:
        print("❌ Critical Error: Master assembly array tracking is empty.")

print("✅ Pipeline functions compiled and ready for deployment blocks below.")

---
## 🎬 Native long-form generation (YouTube-pace fix)
`generate_long_voiceover` above splits the script into isolated sentences and glues
them with a fixed silence gap — every sentence loses all context from its neighbors,
which is what produces the flat, disjointed pacing. OmniVoice's own `generate()` has
built-in long-form handling: it chunks at ~15s punctuation-aware boundaries (not
per-sentence), **crossfades** chunks instead of hard-cutting them, and — critically —
uses the first generated chunk as the voice reference for every later chunk, keeping
prosody continuous across the whole script. The function below calls `generate()` once
with the full narration and lets the library do this natively, instead of
reimplementing a worse version of it by hand. `generate_long_voiceover` is left as-is
above in case this needs to be compared against or rolled back.

In [ ]:
def generate_native(model_instance, script_text, ref_audio_path=None, output_filename="/kaggle/working/native_voiceover.wav", speed=1.0, num_steps=64):
    """
    Single call into OmniVoice's own long-form path -- no manual sentence
    splitting, no manual silence stitching. The library chunks at punctuation
    boundaries (~15s target) and crossfades chunks itself, using the first
    chunk as an implicit voice reference for the rest when none is given.
    """
    text = script_text.strip()
    if not text:
        print("❌ Error: Script content appears to be empty.")
        return

    kwargs = {
        "text": text,
        "num_step": int(num_steps),
        "speed": float(speed),
    }
    if ref_audio_path and os.path.exists(ref_audio_path):
        kwargs["ref_audio"] = ref_audio_path

    print(f"🎙️ Generating full narration natively ({len(text.split())} words)...")
    audio = model_instance.generate(**kwargs)
    audio_data = audio[0]
    if torch.is_tensor(audio_data):
        audio_data = audio_data.detach().cpu().numpy()
    audio_data = audio_data.flatten()

    sf.write(output_filename, audio_data, 24000)
    print(f"🎉 Generation Completed! File stored at: {output_filename}")

print("✅ Native long-form function compiled.")

---
## 🌐 REST API — Kaggle as a TTS backend
Exposes `generate_long_voiceover` above over a fixed HTTPS URL (via a reserved ngrok
domain, so the address never changes between Kaggle sessions). The pipeline sends the
script text plus a base64-encoded reference clip on every request — this cell holds no
default voice or default script; those choices live on the caller's side.

In [ ]:
import base64
import sys
import threading
import uuid

from fastapi import FastAPI
from fastapi.responses import Response
from pydantic import BaseModel
import nest_asyncio
import uvicorn
from pyngrok import conf, ngrok

# --- Kaggle Secrets: NGROK_AUTHTOKEN + NGROK_DOMAIN --------------------
# Set these once under Kaggle notebook -> Add-ons -> Secrets:
#   NGROK_AUTHTOKEN = <your ngrok authtoken>
#   NGROK_DOMAIN    = <your reserved static domain, e.g. xxxx.ngrok-free.dev>
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    NGROK_AUTHTOKEN = _secrets.get_secret("NGROK_AUTHTOKEN")
    NGROK_DOMAIN = _secrets.get_secret("NGROK_DOMAIN")
except Exception:
    NGROK_AUTHTOKEN = os.environ.get("NGROK_AUTHTOKEN")
    NGROK_DOMAIN = os.environ.get("NGROK_DOMAIN")

if not NGROK_AUTHTOKEN or not NGROK_DOMAIN:
    raise RuntimeError(
        "Set NGROK_AUTHTOKEN and NGROK_DOMAIN via Kaggle Secrets (Add-ons -> Secrets) "
        "before running this cell."
    )

app = FastAPI()

# Progress tracking: generate_native is a single call into the library's own
# long-form path, so there's no per-chunk print to key off anymore (that
# only existed because of the old manual per-sentence loop). We still flip
# a note between starting/running/done so the pipeline UI can show *something*
# is happening, but there's no N/M count available from this call.
_PROGRESS = {"current": 0, "total": 0, "note": ""}
_progress_lock = threading.Lock()


class GenerateRequest(BaseModel):
    text: str
    speed: float = 1.0
    num_steps: int = 64
    ref_audio_b64: str | None = None  # base64-encoded reference clip; omit for Auto Voice


@app.get("/health")
def health():
    return {"status": "ok", "device": device}


@app.get("/progress")
def progress():
    with _progress_lock:
        return dict(_PROGRESS)


@app.post("/generate")
def generate_endpoint(req: GenerateRequest):
    ref_path = None
    if req.ref_audio_b64:
        ref_path = f"/kaggle/working/ref_{uuid.uuid4().hex}.mp3"
        with open(ref_path, "wb") as f:
            f.write(base64.b64decode(req.ref_audio_b64))

    out_path = f"/kaggle/working/tts_{uuid.uuid4().hex}.wav"
    with _progress_lock:
        _PROGRESS.update(current=0, total=1, note="generating (single native call)")
    try:
        generate_native(
            model_instance=model,
            script_text=req.text,
            ref_audio_path=ref_path,
            output_filename=out_path,
            speed=req.speed,
            num_steps=req.num_steps,
        )
    finally:
        with _progress_lock:
            _PROGRESS.update(current=1, note="done")
    with open(out_path, "rb") as f:
        audio_bytes = f.read()
    return Response(content=audio_bytes, media_type="audio/wav")


conf.get_default().auth_token = NGROK_AUTHTOKEN
public_url = ngrok.connect(8000, domain=NGROK_DOMAIN)
print(f"🔗 OmniVoice API live at: {public_url}")
print(f"   Health check: {public_url}/generate  (POST)  |  {public_url}/health (GET)  |  {public_url}/progress (GET)")
print(f"   Put this in pipeline/.env -> OMNIVOICE_BASE={public_url}")

nest_asyncio.apply()
config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
threading.Thread(target=server.run, daemon=True).start()
print("✅ API server running. Keep this cell's kernel alive to keep the backend up.")